# 06 — Statistical Analysis, Human Spot-Check & Publication Paper Artifacts

Generates all camera-ready assets into `$DRIVE_ROOT/paper_artifacts/`:
- **4 LaTeX Tables**: `tab_when2speak.tex`, `tab_pagemantle.tex`, `tab_online_evolution.tex`, `tab_ablations.tex`
- **5 Vector PDF Figures**: `fig_fir_mir_pareto.pdf`, `fig_online_learning_curve.pdf`, `fig_cost_vs_score.pdf`, `fig_ablation_delta.pdf`, `fig_prompt_length_o1.pdf`
- **Human Judge Calibration**: `human_calibration_30.csv` + Spearman $\rho$ / Cohen's $\kappa$ calculator
- **3 Qualitative Case Studies**: Silence Trap, High-$U_t$ De-escalation, DB-CSE Boundary Transfer

In [ ]:
# === CELL 0: Mount Google Drive & Configure Persistent Workspace ===
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/when2speak_research")
else:
    DRIVE_ROOT = Path("./local_drive_mirror").resolve()

for sub in [
    "secrets",
    "config_cache",
    "datasets/pagemantle",
    "datasets/when2speak",
    "datasets/pagemantle_ext",
    "caches",
    "runs",
    "paper_artifacts/tables",
    "paper_artifacts/figures",
]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["DIAL_SKILL_DRIVE_ROOT"] = str(DRIVE_ROOT)
print("Persistent DRIVE_ROOT:", DRIVE_ROOT)


In [ ]:
# === CELL 1: Sync GitHub Repo (git clone / git pull) & Hot-Reload ===
REPO_URL = "https://github.com/BoldHu/multiagent-speak.git"  # Edit once with your repo URL
REPO_DIR = Path("/content/when2speak") if IN_COLAB else Path(".").resolve()

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}
    !pip install -q -e . --no-deps

import importlib, sys, types

# Colab Python 3.12+/3.13 removed stdlib `imp`, which older IPython autoreload still imports.
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp_shim = types.ModuleType("imp")
        _imp_shim.reload = importlib.reload
        sys.modules["imp"] = _imp_shim

try:
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("load_ext", "autoreload")
        _ip.run_line_magic("autoreload", "2")
except Exception as _exc:
    print(f"Skipping autoreload ({_exc}); restart runtime after git pull if needed.")


In [ ]:
# === CELL 2: Bootstrap Secrets & Resolved Model Config (Never Prints Raw Keys) ===
import json
from dial_skill.utils.secrets import describe_secret_sources
from dial_skill.llm.preflight import load_resolved_models, resolve_model_for_role
from dial_skill.config import load_models_config, load_dial_skill_config, load_experiments_config

print("Secret lookup diagnostics:", json.dumps(describe_secret_sources(drive_root=DRIVE_ROOT), indent=2))
MODELS_CFG = load_models_config()
DIAL_CFG = load_dial_skill_config()
EXP_CFG = load_experiments_config()
RESOLVED_MODELS = load_resolved_models(drive_root=DRIVE_ROOT)
print("Resolved models:", RESOLVED_MODELS.model_dump() if RESOLVED_MODELS else "Not yet generated (run Notebook 00)")


## Step 1: Load Experiment Summaries from Google Drive

In [ ]:
MODE = "pilot"  # Switch to "main" / "full" once full sweeps complete
TABLES_DIR = DRIVE_ROOT / "paper_artifacts" / "tables"
FIGS_DIR = DRIVE_ROOT / "paper_artifacts" / "figures"

def _load_json_if_exists(path: Path, default):
    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else default

w2s_summary = _load_json_if_exists(DRIVE_ROOT / "runs" / "main_when2speak" / MODE / "metrics_summary.json", {})
pm_pre = _load_json_if_exists(DRIVE_ROOT / "runs" / "main_pagemantle" / MODE / "pre_evolution_metrics.json", {})
pm_post = _load_json_if_exists(DRIVE_ROOT / "runs" / "main_pagemantle" / MODE / "post_evolution_metrics.json", {})
abl_summary = _load_json_if_exists(DRIVE_ROOT / "runs" / "ablations_a1_a7" / MODE / "ablation_summary.json", {})
pareto_rows = _load_json_if_exists(DRIVE_ROOT / "runs" / "scaling_pareto" / MODE / "lambda_pareto.json", [])
print("Loaded summaries -> W2S:", bool(w2s_summary), "| PM:", bool(pm_pre), "| Ablations:", bool(abl_summary), "| Pareto:", len(pareto_rows))


## Step 2: Generate Publication LaTeX Tables (`paper_artifacts/tables/*.tex`)

In [ ]:
from dial_skill.eval.report import (
    generate_tab_ablations,
    generate_tab_online_evolution,
    generate_tab_pagemantle,
    generate_tab_when2speak,
)

if w2s_summary.get("metrics"):
    w2s_rows = [{"method": k, "model": "Luna/Terra", **v} for k, v in w2s_summary["metrics"].items()]
    print("Wrote:", generate_tab_when2speak(w2s_rows, TABLES_DIR / "tab_when2speak.tex"))

pm_combined = {**pm_pre, **pm_post}
if pm_combined:
    pm_rows = [{"method": k, **v} for k, v in pm_combined.items()]
    print("Wrote:", generate_tab_pagemantle(pm_rows, TABLES_DIR / "tab_pagemantle.tex"))

if abl_summary:
    abl_rows = [
        {
            "variant": vid,
            "w2s_macro_f1": data["when2speak"].get("macro_f1"),
            "w2s_fir": data["when2speak"].get("fir"),
            "pm_overall": data["pagemantle"].get("overall_score"),
            "pm_audience": data["pagemantle"].get("audience_and_location"),
            "token_reduction_pct": data["when2speak"].get("token_reduction_pct"),
        }
        for vid, data in abl_summary.items()
    ]
    print("Wrote:", generate_tab_ablations(abl_rows, TABLES_DIR / "tab_ablations.tex"))


## Step 3: Generate Publication Vector PDF Figures (`paper_artifacts/figures/*.pdf`)

In [ ]:
from dial_skill.eval.report import (
    plot_ablation_delta,
    plot_cost_vs_score,
    plot_fir_mir_pareto,
    plot_online_learning_curve,
    plot_prompt_length_o1,
)

if pareto_rows:
    refs = [{"label": "When2Speak SFT (Ref)", "fir": 0.04, "mir": 0.50}, {"label": "When2Speak RL (Ref)", "fir": 0.06, "mir": 0.20}]
    print("Wrote:", plot_fir_mir_pareto(pareto_rows, refs, FIGS_DIR / "fig_fir_mir_pareto.pdf"))

if pm_combined:
    pts = [
        {"label": k, "cost_per_task_usd": v.get("cost_per_task_usd", 0.0), "overall_score": v.get("overall_score", 0.0)}
        for k, v in pm_combined.items()
    ]
    print("Wrote:", plot_cost_vs_score(pts, FIGS_DIR / "fig_cost_vs_score.pdf"))

if abl_summary and "Full_DIAL_Skill" in abl_summary:
    base_f1 = abl_summary["Full_DIAL_Skill"]["when2speak"]["macro_f1"]
    deltas = [
        {"label": vid, "delta": data["when2speak"]["macro_f1"] - base_f1}
        for vid, data in abl_summary.items()
        if vid != "Full_DIAL_Skill"
    ]
    print("Wrote:", plot_ablation_delta(deltas, FIGS_DIR / "fig_ablation_delta.pdf"))


## Step 4: Export 30-Sample Human Spot-Check CSV & Auto-Extract 3 Case Studies

In [ ]:
import pandas as pd
from dial_skill.eval.judge import compute_cohens_kappa, compute_spearman_rho, export_human_spotcheck_csv
from dial_skill.eval.report import extract_paper_case_studies, save_case_studies
from dial_skill.schemas import EpisodeResult, JudgeRubricScore, PageMantleTask
from dial_skill.utils.checkpoint import load_jsonl

pm_pred_file = DRIVE_ROOT / "runs" / "main_pagemantle" / MODE / "preds_DIAL_Skill_Evolved_Full.jsonl"
pm_judge_file = DRIVE_ROOT / "runs" / "main_pagemantle" / MODE / "judge_DIAL_Skill_Evolved_Full.jsonl"
pm_eval_file = DRIVE_ROOT / "datasets" / "pagemantle_ext" / "pm_eval_32.jsonl"
evo_log_file = DRIVE_ROOT / "runs" / "online_dbcse_waves" / MODE / "dbcse_full" / "skill_evolution_log.jsonl"

if pm_pred_file.is_file() and pm_eval_file.is_file():
    eps = [EpisodeResult.model_validate(r) for r in load_jsonl(pm_pred_file)]
    tasks = [PageMantleTask.model_validate_json(l) for l in pm_eval_file.read_text(encoding="utf-8").splitlines() if l.strip()]
    j_scores = [JudgeRubricScore.model_validate(r) for r in load_jsonl(pm_judge_file)] if pm_judge_file.is_file() else []

    csv_path = export_human_spotcheck_csv(
        episodes=eps, tasks=tasks, judge_scores=j_scores,
        out_csv=DRIVE_ROOT / "paper_artifacts" / "human_calibration_30.csv", n_samples=30, seed=EXP_CFG["seed"],
    )
    print("Exported blinded 30-sample human calibration CSV:", csv_path)

    evo_log = load_jsonl(evo_log_file) if evo_log_file.is_file() else []
    cases = extract_paper_case_studies(episodes=eps, tasks=tasks, evolution_log=evo_log)
    case_path = save_case_studies(cases, DRIVE_ROOT / "paper_artifacts" / "case_studies.json")
    print("Saved 3 paper case studies:", case_path)

# After filling in `human_overall_0_10` and `human_timing_binary_ok` in human_calibration_30.csv, re-run:
cal_csv = DRIVE_ROOT / "paper_artifacts" / "human_calibration_30.csv"
if cal_csv.is_file():
    df = pd.read_csv(cal_csv).dropna(subset=["human_overall_0_10", "judge_overall_0_10"])
    if len(df) >= 2:
        rho = compute_spearman_rho(df["judge_overall_0_10"].tolist(), df["human_overall_0_10"].tolist())
        print("Human vs Blinded Dual-Judge Spearman rho:", rho)
        if "human_timing_binary_ok" in df.columns and df["human_timing_binary_ok"].notna().all():
            judge_bin = (df["judge_overall_0_10"] >= 6.0).astype(int).tolist()
            human_bin = df["human_timing_binary_ok"].astype(int).tolist()
            print("Human vs Blinded Dual-Judge Cohen's kappa:", compute_cohens_kappa(judge_bin, human_bin))
